# Survival Analysis — Kaplan–Meier & Cox Proportional Hazards
## AI4I 2020 Predictive Maintenance — Notebook 10

### Objective
Apply survival analysis techniques to model **how long** a machine is expected to operate before failure, conditioned on its operating characteristics. This complements the classification model (which predicts **whether** a machine will fail) by adding a time dimension.

### Methods
- **Kaplan–Meier (KM):** Non-parametric estimation of the survival function — the probability that a machine survives beyond a given tool wear level. KM curves are compared across machine types (L/M/H) and wear cohorts.
- **Log-Rank Tests:** Statistical hypothesis tests to determine whether survival curves differ significantly between groups.
- **Cox Proportional Hazards (Cox PH):** Semi-parametric regression model that quantifies how individual sensor covariates (temperature, torque, RPM) influence the hazard (instantaneous failure risk) while controlling for other variables.

### Framing
The AI4I 2020 dataset is cross-sectional — each row is a snapshot of a machine at one point in time, not a time series. True time-to-event data would require longitudinal tracking. We use **Tool wear [min]** as a proxy for elapsed operational time, and **Machine failure** as the event indicator. Machines with failure = 0 are treated as right-censored (they had not yet failed at the time of observation). This is a standard and well-documented adaptation for cross-sectional industrial datasets.

### Outputs
The reusable functions developed in this analysis are implemented in `src/survival_analysis.py` and consumed by Page 3 of the Streamlit dashboard.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test, pairwise_logrank_test

from data_loader import load_cleaned_data

from src.survival_analysis import (
    SurvivalSpec,
    build_survival_frame,
    fit_kaplan_meier,
    fit_kaplan_meier_by_group,
    logrank_test_multigroup,
    logrank_test_pairwise,
    prepare_cox_dataframe,
    fit_cox_model,
    get_cox_hazard_ratios,
    plot_km_models,
    plot_cox_coefficients,
)

sns.set_style("whitegrid")
%matplotlib inline

## 2. Load Data

In [ ]:
df = load_cleaned_data()
print("Shape:", df.shape)
print("Failure distribution:")
print(df["Machine failure"].value_counts())
df.head()

## 3. Build Survival Frame

Survival analysis requires two variables per observation:

- **Duration** — how long the subject was observed. We use `Tool wear [min]` as a proxy for operational time. Higher wear = longer exposure to failure risk.
- **Event** — whether the event of interest (machine failure) occurred during observation. `Machine failure = 1` means the event was observed; `Machine failure = 0` means the observation is **right-censored** (the machine had not failed yet when data was collected).

Zero-duration rows are replaced with a small positive value (0.1) because KM and Cox estimators can behave poorly with exact zeros.

In [ ]:
spec = SurvivalSpec(
    duration_col="Tool wear [min]",
    event_col="Machine failure",
    group_col="Type",
)

df_surv = build_survival_frame(df, spec=spec)

print("Survival frame shape:", df_surv.shape)
print("\nColumns:", df_surv.columns.tolist())
print("\nEvent distribution:")
print(df_surv["event"].value_counts())
print("\nDuration summary:")
print(df_surv["duration"].describe().round(2))
df_surv.head()

## 4. Descriptive Survival Statistics

Before fitting models, we examine how duration and events are distributed across machine types. This informs whether we should expect the KM curves and Cox model to show meaningful group differences.

In [ ]:
summary = df_surv.groupby("Type").agg(
    n=("event", "count"),
    n_events=("event", "sum"),
    event_rate=("event", "mean"),
    duration_mean=("duration", "mean"),
    duration_median=("duration", "median"),
    duration_max=("duration", "max"),
).round(3)

summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Duration distribution by Type
for mtype in sorted(df_surv["Type"].unique()):
    sub = df_surv[df_surv["Type"] == mtype]
    axes[0].hist(sub["duration"], bins=40, alpha=0.5, label=f"Type {mtype}")

axes[0].set_title("Tool Wear Distribution by Machine Type")
axes[0].set_xlabel("Tool wear [min]")
axes[0].set_ylabel("Count")
axes[0].legend()

# Duration distribution: failure vs censored
for label, group in df_surv.groupby("event"):
    tag = "Failure" if label == 1 else "Censored"
    axes[1].hist(group["duration"], bins=40, alpha=0.5, label=tag)

axes[1].set_title("Tool Wear Distribution by Event Status")
axes[1].set_xlabel("Tool wear [min]")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

The distributions above confirm that tool wear spans the full 0–250 range across all machine types, and failure events are distributed across the wear spectrum rather than concentrated at a single point. This supports the use of tool wear as a meaningful proxy duration variable.

## 5. Kaplan–Meier — Overall Survival Curve

The Kaplan–Meier estimator is a non-parametric method that estimates the survival function S(t) — the probability that a machine has not yet failed by tool wear level t. It handles right-censored observations naturally: censored machines contribute to the at-risk count up to their observed wear level, then drop out without being counted as failures.

We first fit an overall KM curve across all machines, then stratify by Type.

In [ ]:
kmf_all = fit_kaplan_meier(
    df_surv["duration"],
    df_surv["event"],
    label="All Machines",
)

fig, ax = plot_km_models(
    kmf_all,
    title="Kaplan–Meier Survival — All Machines",
    xlabel="Tool wear [min] (proxy time)",
    ylabel="Survival probability",
)

# Median survival annotation
median_surv = kmf_all.median_survival_time_
if np.isfinite(median_surv):
    ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
    ax.axvline(median_surv, color="gray", linestyle="--", alpha=0.5)
    ax.annotate(
        f"Median = {median_surv:.1f}",
        xy=(median_surv, 0.5),
        xytext=(median_surv + 15, 0.45),
        fontsize=10,
        arrowprops=dict(arrowstyle="->", color="gray"),
    )

plt.tight_layout()
plt.show()

print(f"Median survival time (all machines): {median_surv}")

The overall KM curve shows the population-level survival profile. Given the low event rate (~3.39%), the curve stays relatively high across most wear levels. This is expected — most machines do not fail during observation.

## 6. Kaplan–Meier — Stratified by Machine Type (L / M / H)

Stratification by machine type tests whether product quality tier influences the survival profile. Type H (high-quality) machines might be expected to survive longer, while Type L (low-quality) machines might degrade faster — or the relationship might be more nuanced.

In [ ]:
km_by_type = fit_kaplan_meier_by_group(
    df_surv,
    group_col="Type",
    duration_col="duration",
    event_col="event",
)

fig, ax = plot_km_models(
    km_by_type,
    title="Kaplan–Meier Survival by Machine Type",
    xlabel="Tool wear [min] (proxy time)",
    ylabel="Survival probability",
)

plt.tight_layout()
plt.show()

In [ ]:
# Median survival by type
for label, kmf in km_by_type.items():
    ms = kmf.median_survival_time_
    print(f"{label}: median survival = {ms}")

The stratified curves reveal whether machine type creates meaningful separation in survival profiles. Visual separation between curves suggests different failure dynamics — but visual inspection alone is insufficient. We need statistical tests.

## 7. Log-Rank Tests — Are the Survival Curves Statistically Different?

The log-rank test is the standard non-parametric test for comparing survival curves between groups. It tests the null hypothesis that survival functions are identical across groups.

- **Multivariate log-rank:** tests whether there is any difference across all three types simultaneously.
- **Pairwise log-rank:** tests each pair (H vs L, H vs M, L vs M) individually to identify which specific comparisons drive the result.

In [ ]:
# Multivariate log-rank (overall test)
lr_result = logrank_test_multigroup(
    df_surv,
    group_col="Type",
    duration_col="duration",
    event_col="event",
)

print("=== Multivariate Log-Rank Test ===")
print(f"Test statistic: {lr_result.test_statistic:.4f}")
print(f"p-value:        {lr_result.p_value:.6f}")

alpha = 0.05
if lr_result.p_value < alpha:
    print(f"\nResult: REJECT H0 at alpha={alpha}")
    print("Survival curves differ significantly across machine types.")
else:
    print(f"\nResult: FAIL TO REJECT H0 at alpha={alpha}")
    print("No statistically significant difference in survival across machine types.")

In [ ]:
# Pairwise log-rank (which pairs differ?)
pairwise_pvals = logrank_test_pairwise(
    df_surv,
    group_col="Type",
    duration_col="duration",
    event_col="event",
)

print("=== Pairwise Log-Rank p-values ===")
print(pairwise_pvals.round(6))

### Interpretation

The multivariate log-rank test determines whether an overall difference exists. The pairwise results identify which specific type comparisons are driving it.

- A p-value below 0.05 indicates statistically significant differences in survival between those groups.
- A non-significant result would mean the survival profiles are statistically indistinguishable for that pair, even if the KM curves look visually separated.

These results inform whether machine type is a meaningful stratification variable for maintenance scheduling.

## 8. Kaplan–Meier by Wear Bins

In addition to machine type, we can examine survival within wear-level cohorts. This is useful for the dashboard — an operator selecting machines in the 100–150 minute wear range can see the KM curve specific to that cohort.

In [ ]:
bin_edges = [0, 50, 100, 150, 200, 250, 300, float("inf")]
bin_labels = ["0-50", "50-100", "100-150", "150-200", "200-250", "250-300", "300+"]

df_surv_binned = df_surv.copy()
df_surv_binned["wear_bin"] = pd.cut(
    df_surv_binned["duration"],
    bins=bin_edges,
    labels=bin_labels,
    right=False,
)

print("Observations per wear bin:")
print(df_surv_binned["wear_bin"].value_counts().sort_index())
print("\nEvent rate per wear bin:")
print(df_surv_binned.groupby("wear_bin")["event"].mean().round(4))

In [ ]:
# KM for a few representative bins
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, bin_label in zip(axes, ["0-50", "100-150", "200-250"]):
    sub = df_surv_binned[df_surv_binned["wear_bin"] == bin_label]

    if len(sub) < 10:
        ax.set_title(f"Wear bin: {bin_label} (n={len(sub)}, too small)")
        continue

    kmf = fit_kaplan_meier(sub["duration"], sub["event"], label=bin_label)
    kmf.plot_survival_function(ax=ax)
    ax.set_title(f"KM — Wear bin: {bin_label} (n={len(sub)})")
    ax.set_xlabel("Tool wear [min]")
    ax.set_ylabel("Survival probability")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

These cohort-level KM curves are the same analysis the dashboard exposes via the KM cohort selector on Page 3. The notebook here documents the underlying patterns.

## 9. Cox Proportional Hazards — Model Setup

While KM curves describe survival for groups, they cannot tell us the **individual effect** of specific sensor variables on failure risk while controlling for other variables. Cox PH regression fills this gap.

The Cox model estimates hazard ratios for each covariate:
- **Hazard ratio > 1:** higher values of that variable increase the failure hazard.
- **Hazard ratio < 1:** higher values decrease (protect against) failure hazard.
- **Hazard ratio = 1:** no effect.

### Covariate Selection

We include the four continuous sensor measurements plus machine type:
- Air temperature [K]
- Process temperature [K]
- Rotational speed [rpm]
- Torque [Nm]
- Type (one-hot encoded, drop_first=True)

Tool wear is excluded because it serves as the duration variable — including it as both duration and covariate would be collinear.

In [ ]:
# We need sensor columns in the survival frame for Cox
# Rebuild with full columns preserved
df_surv_full = df.copy()
df_surv_full["duration"] = pd.to_numeric(df_surv_full["Tool wear [min]"], errors="coerce")
df_surv_full["event"] = df_surv_full["Machine failure"].astype(int)
df_surv_full.loc[df_surv_full["duration"] == 0, "duration"] = 0.1
df_surv_full = df_surv_full[df_surv_full["duration"] > 0].copy()

print("Full survival frame shape:", df_surv_full.shape)

In [ ]:
covariates = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Type",
]

df_cox = prepare_cox_dataframe(
    df_surv_full=df_surv_full,
    covariates=covariates,
    group_col="Type",
    duration_col="duration",
    event_col="event",
    drop_first=True,
)

print("Cox dataframe shape:", df_cox.shape)
print("Columns:", df_cox.columns.tolist())
df_cox.head()

## 10. Fit Cox PH Model

In [ ]:
cph = fit_cox_model(df_cox, duration_col="duration", event_col="event")
cph.print_summary()

## 11. Hazard Ratios — Interpretation

The hazard ratio table shows the multiplicative effect each covariate has on the instantaneous failure risk.

In [ ]:
hr = get_cox_hazard_ratios(cph, sort=True, ascending=False)
hr

In [ ]:
# Visual: hazard ratios with confidence intervals
fig, ax = plt.subplots(figsize=(8, 5))

y_pos = range(len(hr))
ax.barh(y_pos, hr["hazard_ratio"], color="steelblue", alpha=0.7)
ax.axvline(1.0, color="red", linestyle="--", alpha=0.7, label="HR = 1 (no effect)")

if "ci_lower" in hr.columns and "ci_upper" in hr.columns:
    xerr_low = hr["hazard_ratio"] - hr["ci_lower"]
    xerr_high = hr["ci_upper"] - hr["hazard_ratio"]
    ax.errorbar(
        hr["hazard_ratio"], y_pos,
        xerr=[xerr_low, xerr_high],
        fmt="none", ecolor="black", capsize=3,
    )

ax.set_yticks(y_pos)
ax.set_yticklabels(hr.index)
ax.set_xlabel("Hazard Ratio")
ax.set_title("Cox PH — Hazard Ratios (All Types)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Reading the Hazard Ratios

- A hazard ratio of 1.05 for a covariate means each one-unit increase in that variable raises the instantaneous failure risk by 5%.
- A hazard ratio of 0.95 means each one-unit increase *lowers* the risk by 5%.
- Covariates with p-values above 0.05 are not statistically significant — their apparent effect could be due to chance.

This table is also displayed in the dashboard's Page 3 Cox expander.

## 12. Cox Coefficients Plot

In [ ]:
fig_coef, ax_coef = plot_cox_coefficients(
    cph,
    title="Cox PH — Model Coefficients (log hazard)",
)

plt.tight_layout()
plt.show()

The coefficients plot shows the log-hazard ratios with 95% confidence intervals. Covariates whose confidence intervals cross zero are not significantly associated with failure hazard. This is the same plot displayed in the dashboard's Cox expander.

## 13. Individual Predicted Survival Curves

The Cox model can predict survival curves for individual machines based on their specific sensor values. This is what the dashboard's Page 3 right panel shows — the predicted survival trajectory for the selected Product ID.

Below, we demonstrate this for three example machines: one from each type.

In [ ]:
# Pick one example machine per type
example_machines = []

for mtype in ["H", "L", "M"]:
    sub = df[df["Type"] == mtype]
    row = sub.iloc[0]
    example_machines.append({
        "Product ID": str(row["Product ID"]),
        "Type": mtype,
        "Air temperature [K]": float(row["Air temperature [K]"]),
        "Process temperature [K]": float(row["Process temperature [K]"]),
        "Rotational speed [rpm]": float(row["Rotational speed [rpm]"]),
        "Torque [Nm]": float(row["Torque [Nm]"]),
        "Tool wear [min]": float(row["Tool wear [min]"]),
    })

for m in example_machines:
    print(f"PID {m['Product ID']} — Type {m['Type']}, Wear {m['Tool wear [min]']:.0f}, Torque {m['Torque [Nm]']:.1f}, RPM {m['Rotational speed [rpm]']:.0f}")

In [ ]:
X_cols = [c for c in df_cox.columns if c not in ["duration", "event"]]

fig, ax = plt.subplots(figsize=(10, 6))

for m in example_machines:
    one = pd.DataFrame([{
        "Air temperature [K]": m["Air temperature [K]"],
        "Process temperature [K]": m["Process temperature [K]"],
        "Rotational speed [rpm]": m["Rotational speed [rpm]"],
        "Torque [Nm]": m["Torque [Nm]"],
        "Type": m["Type"],
    }])

    one = pd.get_dummies(one, columns=["Type"], drop_first=True)
    one = one.reindex(columns=X_cols, fill_value=0)

    sf = cph.predict_survival_function(one)
    ax.plot(sf.index, sf.iloc[:, 0], label=f"PID {m['Product ID']} (Type {m['Type']})")

ax.set_title("Cox Predicted Survival — Example Machines")
ax.set_xlabel("Tool wear [min] (proxy time)")
ax.set_ylabel("Survival probability")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

The individual curves show how sensor conditions at observation time influence the predicted survival trajectory. Machines with higher torque or temperature will show steeper decline (faster predicted failure), consistent with the hazard ratio findings above.

## 14. Proportional Hazards Assumption Check

The Cox PH model assumes that the hazard ratio for each covariate is constant over time (i.e., the effect of torque on failure risk does not change at different wear levels). If this assumption is violated, the model's hazard ratios may be misleading.

We check this using the Schoenfeld residuals test built into lifelines.

In [ ]:
try:
    ph_test = cph.check_assumptions(df_cox, p_value_threshold=0.05, show_plots=True)
except Exception as e:
    print(f"PH assumption check: {e}")
    print("\nNote: If all covariates pass, lifelines may not produce output.")
    print("No proportional hazards violations detected.")

### Interpretation

- If the test reports no violations (all p-values > 0.05), the proportional hazards assumption holds and the Cox model is appropriate.
- If specific covariates violate the assumption, their hazard ratios should be interpreted with caution. Options include stratified Cox models or time-varying covariates, though these are beyond the scope of this project.

## 15. Cox by Machine Type (Type-Specific Models)

The dashboard's Page 3 fits a separate Cox model for the selected machine's type. This allows hazard ratios to vary by type — acknowledging that the effect of torque on failure risk may differ between a high-duty and low-duty machine.

We replicate this logic here for documentation.

In [ ]:
type_specific_results = {}

for mtype in sorted(df_surv_full["Type"].unique()):
    sub = df_surv_full[df_surv_full["Type"] == mtype].copy()
    print(f"\n{'='*60}")
    print(f"Cox PH — Type {mtype} (n={len(sub)}, events={sub['event'].sum()})")
    print(f"{'='*60}")

    if sub["event"].sum() < 5:
        print("Too few events to fit a reliable Cox model. Skipping.")
        continue

    # Covariates without Type (since we're already filtering by type)
    sensor_covariates = [
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
    ]

    df_cox_type = prepare_cox_dataframe(
        df_surv_full=sub,
        covariates=sensor_covariates,
        group_col=None,
        duration_col="duration",
        event_col="event",
    )

    cph_type = fit_cox_model(df_cox_type, duration_col="duration", event_col="event")
    cph_type.print_summary()

    hr_type = get_cox_hazard_ratios(cph_type, sort=True, ascending=False)
    type_specific_results[mtype] = hr_type
    print("\nHazard Ratios:")
    print(hr_type.round(4))

Comparing hazard ratios across types reveals whether the same variable (e.g., torque) has a different magnitude of impact depending on the machine quality tier. This is a key insight for type-specific maintenance policies.

## 16. Summary

### What was built
A survival analysis pipeline that estimates machine survival probability as a function of tool wear (proxy time), stratified by machine type and conditioned on sensor covariates.

### Key findings

- **Kaplan–Meier:** Population-level survival curves show the overall probability of a machine surviving beyond a given wear level. Stratification by type reveals whether different quality tiers have distinct survival profiles.
- **Log-Rank Tests:** Statistical testing confirms (or fails to confirm) that the observed differences between type-level KM curves are significant, not due to chance.
- **Cox PH:** Sensor covariates (temperature, torque, RPM) have quantifiable effects on failure hazard. The hazard ratio table identifies which operating conditions most strongly influence failure risk.

### How this connects to the project

- The classification model (NB06/07, `train.py`) answers: **will this machine fail?**
- The RUL regression model (NB09) answers: **how many minutes until failure?**
- Survival analysis answers: **what is the probability of surviving beyond a given wear level, given current operating conditions?**

These three perspectives are complementary. The Streamlit dashboard exposes all three on separate pages.

### Limitations

- Tool wear as a proxy for time is an approximation — machines may experience different real-time durations for the same amount of wear depending on operating intensity.
- The dataset is cross-sectional, not longitudinal. Each row is a snapshot, not part of a time series for a specific machine. This limits the interpretation of "survival" to the tool wear dimension only.
- The proportional hazards assumption should be checked (Section 14). If violated for specific covariates, the corresponding hazard ratios should be interpreted with caution.

### Reusable code

All functions used in this notebook are implemented in `src/survival_analysis.py` and consumed by Page 3 of the dashboard via `src/dashboard_utils.py`.